In [1]:
!pip install -q openai python-dotenv

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")


Enter your OpenAI API key: ··········


In [3]:
from openai import OpenAI

client = OpenAI()

In [16]:
def calculator(expression: str) -> str:
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

In [17]:
SYSTEM_PROMPT = """
You are a helpful AI agent.

You can:
- Answer general questions
- Use a calculator tool when math is required

If math is required:
Respond in JSON format:
{
  "tool": "calculator",
  "input": "expression here"
}

Otherwise:
Respond normally with text.
"""

In [18]:
import json

def run_agent(user_query):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ]
    )

    message = response.choices[0].message.content

    # Try parsing as tool call
    try:
        parsed = json.loads(message)

        if parsed.get("tool") == "calculator":
            tool_result = calculator(parsed["input"])

            # Send tool result back to model
            final_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_query},
                    {"role": "assistant", "content": message},
                    {"role": "tool", "content": tool_result}
                ]
            )

            return final_response.choices[0].message.content

    except:
        return message

In [19]:
print(run_agent("What is the capital of France?"))

The capital of France is Paris.


In [8]:
print(run_agent("What is 234 * 56?"))

{
  "tool": "calculator",
  "input": "234 * 56"
}


In [10]:
!pip install -q openai-agents
from agents import Agent, Runner, function_tool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.6/403.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 12.4 MB/s eta 0:00:00


In [11]:
@function_tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    Example input: "234 * 56"
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
agent = Agent(
    name="SimpleMathAgent",
    instructions="""
You are a helpful AI agent.

- Answer general knowledge questions directly.
- Use the calculator tool whenever mathematical computation is required.
- Always use the tool for arithmetic.
""",
    tools=[calculator],
    model="gpt-4o-mini"
)

In [15]:
result = await Runner.run(
    agent,
    "What is 234 * 56?"
)

print(result.final_output)

The result of \( 234 \times 56 \) is 13,104.


### **Single Financial Analyst Agent**

In [22]:
from openai import OpenAI
from agents import Agent, Runner, function_tool
from typing import Dict
import math

# -----------------------------------
# Tool Definitions
# -----------------------------------

@function_tool
def compound_interest(principal: float, rate: float, years: int) -> Dict:
    """
    Calculate compound interest.
    """
    amount = principal * ((1 + rate) ** years)
    return {
        "principal": principal,
        "rate": rate,
        "years": years,
        "final_amount": round(amount, 2)
    }

@function_tool
def risk_score(volatility: float, beta: float) -> Dict:
    """
    Simple risk scoring.
    """
    score = volatility * beta
    return {
        "volatility": volatility,
        "beta": beta,
        "risk_score": round(score, 3)
    }

# -----------------------------------
# Single Agent
# -----------------------------------

financial_agent = Agent(
    name="Financial Analyst Agent",
    instructions="""
    You are a financial analyst.
    Use tools whenever calculations are required.
    Explain results clearly.
    """,
    tools=[compound_interest, risk_score],
)

# -----------------------------------
# Execution
# -----------------------------------

result = await Runner.run(
    financial_agent,
    """
    If I invest 10000 at 8% for 5 years,
    and volatility is 0.2 and beta is 1.3,
    calculate final amount and risk score.
    """
)

print(result.final_output)

Here are the results:

1. Final Investment Amount: If you invest $10,000 at an 8% annual compound interest rate for 5 years, your final amount will be approximately $14,693.28.

2. Risk Score: With a volatility of 0.2 and a beta of 1.3, the risk score is 0.26. This means the investment has a moderate risk level, reflecting both market sensitivity (beta) and price variation (volatility).

Let me know if you want more detail on either calculation!


### **Multi-Agent Architecture**

In [24]:
from agents import Agent, Runner, function_tool, handoff

# -----------------------------------
# Tools
# -----------------------------------

@function_tool
def compound_interest(principal: float, rate: float, years: int):
    amount = principal * ((1 + rate) ** years)
    return {"final_amount": round(amount, 2)}

@function_tool
def risk_score(volatility: float, beta: float):
    score = volatility * beta
    return {"risk_score": round(score, 3)}

# -----------------------------------
# Specialist Agent
# -----------------------------------

calculation_agent = Agent(
    name="Calculation Agent",
    instructions="""
    You only perform financial calculations.
    Always use tools.
    Return structured data only.
    """,
    tools=[compound_interest, risk_score],
)

# -----------------------------------
# Reviewer Agent
# -----------------------------------

review_agent = Agent(
    name="Review Agent",
    instructions="""
    Review calculation results.
    Explain them clearly in business language.
    Add interpretation and risk commentary.
    """
)

# -----------------------------------
# Planner Agent (Orchestrator)
# -----------------------------------

planner_agent = Agent(
    name="Planner Agent",
    instructions="""
    Break user request into:
    1. Calculation task
    2. Review task

    First handoff to Calculation Agent.
    Then handoff results to Review Agent.
    """
)

# -----------------------------------
# Handoff Chain
# -----------------------------------

planner_agent = handoff(
    planner_agent,
    calculation_agent
)

calculation_agent = handoff(
    calculation_agent,
    review_agent
)

# -----------------------------------
# Execution
# -----------------------------------

result = await Runner.run(
    financial_agent,
    """
    If I invest 10000 at 8% for 5 years,
    and volatility is 0.2 and beta is 1.3,
    calculate final amount and risk score.
    """
)

print(result.final_output)

Here are the results:

1. Final Investment Amount:
- If you invest $10,000 at 8% annual interest for 5 years (compound interest), your investment will grow to $14,693.28.

2. Risk Score:
- With a volatility of 0.2 and a beta of 1.3, the simple risk score for this investment is 0.26. This indicates a moderate risk level, factoring in both the asset's price fluctuations and its sensitivity to market movements.

Let me know if you need further explanation or details!


### **CRAG**

In [29]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken

In [30]:
import os
from getpass import getpass

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

In [31]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings()

In [32]:
documents = [
    Document(page_content="""
    Standard RAG retrieves documents and generates answers from them.
    It may still hallucinate if retrieval is weak.
    """),

    Document(page_content="""
    Corrective RAG evaluates retrieval quality.
    If documents are insufficient, it rewrites the query and re-retrieves.
    """),

    Document(page_content="""
    Retrieval evaluation improves factual reliability in enterprise systems.
    """)
]

In [33]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [34]:
evaluation_prompt = ChatPromptTemplate.from_template("""
You are a retrieval evaluator.

Question:
{question}

Retrieved Context:
{context}

Is the context sufficient to answer the question?

Respond ONLY with:
SUFFICIENT
or
INSUFFICIENT
""")

In [35]:
rewrite_prompt = ChatPromptTemplate.from_template("""
Rewrite this question to improve document retrieval precision.

Original Question:
{question}
""")

In [36]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer strictly using the context below.

Question:
{question}

Context:
{context}

If insufficient information exists, say:
"I do not have enough information."
""")

In [37]:
evaluation_chain = evaluation_prompt | llm
rewrite_chain = rewrite_prompt | llm
answer_chain = answer_prompt | llm

In [38]:
class CRAGState:
    def __init__(self, question):
        self.original_question = question
        self.current_question = question
        self.context = ""
        self.retrieval_evaluation = None
        self.rewritten = False

In [39]:
def retrieve_step(state: CRAGState):
    docs = retriever.invoke(state.current_question)
    state.context = "\n\n".join([d.page_content for d in docs])
    return state

In [40]:
def evaluate_step(state: CRAGState):
    result = evaluation_chain.invoke({
        "question": state.current_question,
        "context": state.context
    })

    state.retrieval_evaluation = result.content.strip()
    return state

In [41]:
def rewrite_step(state: CRAGState):

    if "INSUFFICIENT" in state.retrieval_evaluation:

        result = rewrite_chain.invoke({
            "question": state.original_question
        })

        state.current_question = result.content.strip()
        state.rewritten = True

        # Re-retrieve
        retrieve_step(state)

    return state

In [42]:
def generate_answer_step(state: CRAGState):

    result = answer_chain.invoke({
        "question": state.original_question,
        "context": state.context
    })

    return result.content

In [43]:
def corrective_rag_pipeline(question: str):

    state = CRAGState(question)

    # Initial retrieval
    retrieve_step(state)

    # Evaluate retrieval
    evaluate_step(state)

    print("Retrieval Evaluation:", state.retrieval_evaluation)

    # Correct if needed
    rewrite_step(state)

    if state.rewritten:
        print("Query was rewritten to:", state.current_question)

    # Final answer
    answer = generate_answer_step(state)

    return answer

In [44]:
response = corrective_rag_pipeline(
    "How does corrective RAG improve reliability?"
)

print("\nFinal Answer:\n", response)

Retrieval Evaluation: SUFFICIENT

Final Answer:
 Corrective RAG improves reliability by evaluating retrieval quality and rewriting the query to re-retrieve documents if the initial documents are insufficient. This process enhances factual reliability in enterprise systems, reducing the chances of hallucination that may occur with standard RAG when retrieval is weak.
